# 🄲🄾🅄🅁🅂🄴 🅆🄰🄻🄻🄰🄷 - Telegram Downloader & Cloud Uploader
### ⚡ 100% Telegram Interactive • Progressive Batch & Lecture Sync • 1Gbps Colab Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramjipatel-tech/course-wallah-downloader-bot/blob/main/CourseWallah_Downloader_Colab.ipynb)

---

### 🔑 GITHUB TOKEN SETUP (For Progressive Batch Sync & Private Repos):
1. Generate a **Personal Access Token** at [GitHub Settings -> Personal access tokens (classic)](https://github.com/settings/tokens/new).
2. Note: `CourseWallah Colab`
3. Check the **`repo`** checkbox (Full control of private repositories).
4. Copy the token (`ghp_...`) and paste it into `GITHUB_TOKEN` below.
5. *(Tip)*: To prevent Koyeb website rebuilds during downloading, you can set `GITHUB_REPO` to a dedicated data repo (e.g. `ramjipatel-tech/course-wallah-data`) instead of the main platform repo.


In [ ]:
# @title 🚀 1-CLICK BOT RUNNER (Clone, Setup & Start Bot) { run: "auto" }
import os
import sys
import urllib.request, urllib.error, json
from IPython.display import HTML, display

# ==============================================================================
# 🔑 1. CREDENTIALS FORM
# ==============================================================================
BOT_TOKEN = "7942790938:AAF_YOUR_BOT_TOKEN_HERE"  # @param {type:"string"}
API_ID = 26569107  # @param {type:"integer"}
API_HASH = "8fd4f7c182283cb07ceeb18c21966144"  # @param {type:"string"}
OWNER_ID = 6363840742  # @param {type:"integer"}

# GitHub Personal Access Token (starts with ghp_): Required for syncing batch JSONs
GITHUB_TOKEN = ""  # @param {type:"string"}
# Target repository for synced batch JSONs (default: course-wallah-platform or dedicated data repo)
GITHUB_REPO = "ramjipatel-tech/course-wallah-platform"  # @param {type:"string"}

# Watermark text burned on downloaded videos
WATERMARK_TEXT = "COURSE WALLAH"  # @param {type:"string"}

# ==============================================================================
# ✅ PRE-FLIGHT GITHUB TOKEN VERIFICATION
# ==============================================================================
token = GITHUB_TOKEN.strip() if GITHUB_TOKEN else ""
if token:
    try:
        t_req = urllib.request.Request("https://api.github.com/user", headers={
            "Authorization": f"Bearer {token}",
            "User-Agent": "CourseWallah-Colab/1.0"
        })
        with urllib.request.urlopen(t_req, timeout=10) as t_res:
            u_info = json.loads(t_res.read().decode())
            print(f"✅ GitHub Token Verified! Authenticated as @{u_info.get('login')}")
    except urllib.error.HTTPError as he:
        if he.code == 401:
            display(HTML("""
            <div style="background:#2a0a0a; border:2px solid #ef4444; border-radius:8px; padding:15px; color:#fecaca; margin-top:10px;">
              <h3 style="margin:0 0 10px 0; color:#ef4444;">❌ GITHUB_TOKEN is INVALID (401 Bad credentials)!</h3>
              <p>Your token cannot push batch JSONs to GitHub. To generate a working token:</p>
              <ol>
                <li>Open <a href="https://github.com/settings/tokens/new" target="_blank" style="color:#60a5fa; text-decoration:underline;">GitHub Token Settings (classic)</a></li>
                <li>Enter Note: <b>CourseWallah Colab</b></li>
                <li>Check <b>repo</b> scope</li>
                <li>Click <b>Generate token</b> and paste the <code>ghp_...</code> string above</li>
              </ol>
            </div>
            """))
            raise SystemExit("Invalid GITHUB_TOKEN: Please supply a valid GitHub token with repo scope.")
        else:
            print(f"⚠️ GitHub API response: {he}")

# ==============================================================================
# 📦 2. CLONE REPOSITORY (SUPPORTS PUBLIC & PRIVATE REPOS)
# ==============================================================================
PROJECT_DIR = "/content/course-wallah-downloader-bot"
if not os.path.exists(PROJECT_DIR):
    print("📥 Cloning repository into Colab...")
    if token:
        clone_url = f"https://{token}@github.com/ramjipatel-tech/course-wallah-downloader-bot.git"
    else:
        clone_url = "https://github.com/ramjipatel-tech/course-wallah-downloader-bot.git"
    !git clone {clone_url} {PROJECT_DIR}
else:
    print("🔄 Pulling latest updates from GitHub...")
    !git -C {PROJECT_DIR} pull

if not os.path.exists(PROJECT_DIR):
    display(HTML("""
    <div style="background:#2a0a0a; border:2px solid #ef4444; border-radius:8px; padding:15px; color:#fecaca; margin-top:10px;">
      <h3 style="margin:0 0 10px 0; color:#ef4444;">⚠️ GitHub Repository is PRIVATE!</h3>
      <p>Please enter your GitHub Personal Access Token into the <b>GITHUB_TOKEN</b> field in the form above and run this cell again.</p>
    </div>
    """))
    raise SystemExit("Repository is private. Please provide GITHUB_TOKEN.")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# ==============================================================================
# 🛠️ 3. SYSTEM & PYTHON PACKAGES
# ==============================================================================
print("\n⚙️ Installing FFmpeg, Aria2 & Python dependencies...")
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 mkvtoolnix
!pip install -q pyrofork tgcrypto pysocks requests aiohttp aiofiles cloudscraper beautifulsoup4 urllib3 yt-dlp m3u8 mutagen ffmpeg-python pillow pymupdf pycryptodome cryptography pyaes python-dotenv pytz tqdm colorama flask gunicorn

req_file = os.path.join(PROJECT_DIR, "requirements.txt")
if os.path.exists(req_file):
    !pip install -q -r {req_file}

# ==============================================================================
# 📝 4. WRITE CONFIGURATION (.env)
# ==============================================================================
env_lines = [
    "# Course Wallah Bot Environment",
    f"BOT_TOKEN={BOT_TOKEN.strip()}",
    f"API_ID={API_ID}",
    f"API_HASH={API_HASH.strip()}",
    f"OWNER_ID={OWNER_ID}",
    f"ADMINS={OWNER_ID}",
    f"WATERMARK_TEXT={WATERMARK_TEXT.strip()}",
    f"GITHUB_REPO={GITHUB_REPO.strip()}",
    "GITHUB_BRANCH=main",
    "MAX_UPLOAD_SIZE_BYTES=2097152000",
    "DOWNLOAD_WORKERS=4",
    "UPLOAD_WORKERS=3"
]
if token:
    env_lines.append(f"GITHUB_TOKEN={token}")

with open(os.path.join(PROJECT_DIR, ".env"), "w", encoding="utf-8") as f:
    f.write("\n".join(env_lines) + "\n")

# ==============================================================================
# 🔊 5. ANTI-DISCONNECT KEEP-ALIVE & BOT LAUNCH
# ==============================================================================
display(HTML("""
<div style="background:#121826; border:1px solid #10b981; border-radius:8px; padding:12px; margin-bottom:15px; color:#f3f4f6;">
  <h4 style="margin:0 0 6px 0; color:#10b981;">🟢 Bot Online & Running on Google Colab!</h4>
  <p style="margin:0; font-size:13px;">Progressive Sync Active (auto-pushes every 3 lectures). Send your <code>.txt</code> course file in Telegram.</p>
</div>
<audio loop autoplay style="display:none;">
  <source src="https://raw.githubusercontent.com/anars/blank-audio/master/10-seconds-of-silence.mp3" type="audio/mp3">
</audio>
"""))

print(f"📂 Running bot in: {PROJECT_DIR}")
print("🚀 Launching Course Wallah Downloader Bot...\n")
!python {PROJECT_DIR}/main.py
